<font color="blue">**This notebook is used to train an RL model (specifically PPO) on filter frequency response data to realize a synthesis tool for an arbitrary piecewise-continuous frequency response.**

<font color="blue">The execution flow is very similar to supervised learning models (MLP and SVR) that were developed for this project but with a major difference that, for RL training, data are piecewise continuous frequency response (real world requirements). Initial component values are estimates provided by MLP or SVR. In other words, RL is trained using real world data and inital estimates provided by  suprvisod learning. RL algorithm (PPO in this case) is used to increase the accuracy of MLP estimates by exploring the vast solution space of filter component values

1.  <font color="blue">It requires installation of PySPice library for analysis of a filter.

2. <font color="blue">Number of samples in frequency response is 129.

3. <font color="blue">Filter passband is from 2300MHz to 2400MHz.

4. <font color="blue">Filter frequency response is collected from 1800MHz to 2800MHz.

5. <font color="blue">Inductor values are uH ( micro-Henry) and capacitor values are in pF ( pico-Farad)

6. <font color="blue"> Saved trained MLP is used to synthesize  the component values of the filter as initial estiamtes to PPO.

7. <font color="blue"> Hyperparameter search was done using Optuna library

8. <font color="blue"> File paths shall be updated for the code to run without errors. The current code uses Google drive.

<font color="blue">**Step 1:** First install PySpice, Tensorboard, StableBaseline3 and PyTorch.  

<font color="blue"> File paths shall be updated for the code to run without errors. The current code uses Google drive.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import warnings
# Suppress DeprecationWarnings specifically from jupyter_client
warnings.filterwarnings("ignore", category=DeprecationWarning, module='jupyter_client.*')
# Optionally, suppress all DeprecationWarnings if the above isn't sufficient
warnings.filterwarnings("ignore", category=DeprecationWarning)
import sys
import os


# First installed conda for PySpice installtion
# This could take up to 30 sec
!pip install condacolab
import condacolab
condacolab.install()
!command > /dev/null

# Install ngspice packages, including executables & master package
# This could take 2-3 minutes
!conda install -c conda-forge ngspice-exe --quiet
!conda install -c conda-forge ngspice --quiet
!conda install -c conda-forge ngspice-lib --quiet
# Install pyspice using pip
!pip install -q pyspice
!command > /dev/null

#This is optional: run this to ensure PySpice is installed
!pyspice-post-installation --check-install


# Install tensorboard explicitly in the current conda environment
!pip install tensorboard

<font color="blue">**Step 2:** Import all necessary libraries

In [ ]:
#Import all needed libraries

import numpy as np
import subprocess
import re
import copy
import random
import joblib

from collections import namedtuple
from numpy._core.numerictypes import int32
import pandas as pd
import matplotlib.pyplot as plt
import time


from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.preprocessing   import StandardScaler
from sklearn.preprocessing import QuantileTransformer
from sklearn.base import BaseEstimator, TransformerMixin

import torch
import torch.nn as nn
import torch.optim as optim
from   torch.utils.data import DataLoader, TensorDataset

# Fix: PySpice was installed in python3.11 site-packages, add it to path if not present
import sys
if "/usr/local/lib/python3.11/site-packages" not in sys.path:
    sys.path.append("/usr/local/lib/python3.11/site-packages")
from PySpice.Spice.Netlist import Circuit
from PySpice.Unit import *
from PySpice.Spice.NgSpice import Shared as NgSpiceShared
from PySpice.Spice.NgSpice.Shared import ffi


# Install required RL libraries if not already present
try:
    import gymnasium as gym
    from stable_baselines3 import PPO
    from stable_baselines3.common.monitor import Monitor
except ImportError:
    import subprocess
    import sys
    print("Installing gymnasium and stable-baselines3...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gymnasium", "stable-baselines3"])
    import gymnasium as gym
    from stable_baselines3 import PPO
    from stable_baselines3.common.monitor import Monitor

# Ensure SummaryWriter from torch.utils.tensorboard is available for stable_baselines3
from torch.utils.tensorboard import SummaryWriter

import shutil
import gc
from gymnasium import spaces
from stable_baselines3.common.results_plotter import load_results, ts2xy
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.logger import configure
from stable_baselines3.common.callbacks import BaseCallback


<font color="blue"> **Step 3:** Set config parameters and define methods for filter synthesis and frequency response analysis

In [ ]:
####### CONFIG PARAMETERS #########
# lowest value of frequency response
val_FR_lo=-65
# highest value of frequency response
val_FR_hi=5
# lowest frequency for plot
val_Freq_lo = 1800
# highest frequency for plot
val_Freq_hi = 2800
# number of points in frequency response of network
N_FreqResp=129
FreqVec_MHz=np.linspace(1800,2800,N_FreqResp)

# Identify indices for the passband
passband_idx = np.where((FreqVec_MHz >= 2300) & (FreqVec_MHz <= 2400))[0]

# In-band weight to ensure low IL in the passband and to avoid dominance by OOB rejection
in_band_weight_RL = 100.0


# RL parameters
# Total number of steps in training
Acculamted_steps= 75*8000
# Number of steps before actor/critic reset and use a new sample for training
# For PPO, n_step/Max_steps is number of samples considered for gradient descent update
# Max stpe, essenitally sets how much exploration PPO can do before jumping to a new sample
Max_steps= 64
# Number of samples for error evaluation
N_samples_eval=200
# Total number of steps in hyperpraramter search
Acculamted_steps_HPR= Max_steps*1024/2
# Number of samples for error evaluation in hyperpraramter search
N_samples_eval_HPR=200


#------------------------------------------------------------------------------
# Define method to AC simulate the synthesized network
# It only works for third order filters currenlty
def ACsim(L_nH,Cs_fF,Cp_fF,freqs_MHz, render_mode):
  # scale components
  L_uH =L_nH /1E3
  Cs_pF=Cs_fF/1E3
  Cp_pF=Cp_fF/1E3

  # 1. Define the Circuit
  circuit = Circuit('SynthNet')
  #print(L_uH, Cs_pF, Cp_PF)
  # AC voltage source: Define directly with SPICE string to ensure AC parameter is included
  # Syntax: V<name> <node+> <node-> 'DC <val> AC <val>'
  # add a small resistance to inductors to avoid singularity
  # source
  circuit.V(1,     'nodes',  circuit.gnd, 'DC 0 AC 1')
  # source impedance
  circuit.R(100,   'nodes',  'node1',        50@u_Ohm)
  # node 1 to GND shunt resonator
  circuit.L(0,     'node1',   'n1R',         L_uH[0]  @u_uH)
  circuit.R(800,   'n1R'  ,   'n1mid',       0.000001 @u_Ohm)
  circuit.C(800,   'n1mid',   circuit.gnd,   Cs_pF[0] @u_pF)
  circuit.C(900,   'node1',   circuit.gnd,   Cp_pF[0] @u_pF)
  # node 1 to node 2 series resoantor
  circuit.L(1,     'node1',   'n12R',        L_uH[1]  @u_uH)
  circuit.R(801,   'n12R'  ,  'n12mid',      0.000001 @u_Ohm)
  circuit.C(801,   'n12mid',  'node2',       Cs_pF[1] @u_pF)
  circuit.C(901,   'node1',   'node2',       Cp_pF[1] @u_pF)
  # node 2 to GND shunt resoantor
  circuit.L(2,     'node2',   'n2R',         L_uH[2]  @u_uH)
  circuit.R(802,   'n2R'  ,   'n2mid',       0.000001 @u_Ohm)
  circuit.C(802,   'n2mid',   circuit.gnd,   Cs_pF[2] @u_pF)
  circuit.C(902,   'node2',   circuit.gnd,   Cp_pF[2] @u_pF)
  # load impedance
  circuit.R(200,   'node2',  'nodel',        0.000001@u_Ohm)
  circuit.R(101,   'nodel',   circuit.gnd,   50@u_Ohm)

# Debug: Print the netlist
  #print("--- Generated Netlist ---")
  #print(circuit)

# 2. Setup Ngspice Simulator
# Use ID 0 for the standard system library
  FIXED_NGSPICE_ID = 0
# Check/Reuse existing Ngspice instance to avoid CDefError
  if FIXED_NGSPICE_ID in NgSpiceShared.NgSpiceShared._instances:
      ngspice_shared_instance = NgSpiceShared.NgSpiceShared._instances[FIXED_NGSPICE_ID]
  else:
    # Patch ffi.cdef to ignore duplicate declarations if necessary
    original_cdef = ffi.cdef
    def safe_cdef(csource, *args, **kwargs):
        try:
            original_cdef(csource, *args, **kwargs)
        except Exception as e:
            if "duplicate declaration" in str(e):
                pass
            else:
                raise e
    ffi.cdef = safe_cdef
    try:
        ngspice_shared_instance = NgSpiceShared.NgSpiceShared.new_instance(ngspice_id=FIXED_NGSPICE_ID)
    finally:
        ffi.cdef = original_cdef
# Create the simulator instance linked to the *current* circuit definition
  simulator = circuit.simulator(ngspice_shared=ngspice_shared_instance)

# 3. Run Analysis and Plot
  analysis = simulator.ac(start_frequency=freqs_MHz[0]*1E6@u_Hz, stop_frequency=freqs_MHz[-1]*1E6@u_Hz, number_of_points=np.size(freqs_MHz), variation='lin')
  # pick the load voltage
  v_out_mag = np.array(np.abs(analysis.nodes['nodel']))
  # find the magnitude of frequency response
  MagFreqResp= 20 * np.log10(2*v_out_mag)
  # print( MagFreqResp)
  if render_mode:
    # Plot in dB
    plt.plot(analysis.frequency/1E6, MagFreqResp)
    plt.title("AC Frequency Response")
    plt.xlabel("Frequency (MHz)")
    plt.ylabel("Voltage at Node 2 (dB)")
    plt.grid(True, which="both")
    plt.show()

  # Memory cleanup for ngspice
  ngspice_shared_instance.exec_command('destroy all')
  ngspice_shared_instance.exec_command('remcirc')

  return MagFreqResp

<font color="blue"> **Step 4:** Setup MLP to generate the starting point for RL training:

<font color="blue">1- Custom-made scaling used during MLP training

<font color="blue">2- Pretrained MLP model for filter synthesis that was generated in MLP notebook of this project


<font color="blue"> File paths shall be updated for the code to run without errors. The current code uses Google drive.

In [ ]:
# Define a MLP class for subsequent training
class TunableMLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_layers=3, neurons_per_layer=128):
        super(TunableMLP, self).__init__()

        layers = []

        # Input layer to first hidden layer
        layers.append(nn.Linear(input_dim, neurons_per_layer))
        layers.append(nn.ReLU())

        # Hidden layers
        for _ in range(hidden_layers - 1):
            layers.append(nn.Linear(neurons_per_layer, neurons_per_layer))
            layers.append(nn.ReLU())

        # Output layer
        layers.append(nn.Linear(neurons_per_layer, output_dim))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

# Create a Custom Scaler to apply QuantileTransformer to specific columns and StandardScaler to the rest
class MixedScaler(BaseEstimator, TransformerMixin):
    def __init__(self, quantile_idx, standard_idx):
        self.quantile_idx = quantile_idx
        self.standard_idx = standard_idx
        self.scaler_q = QuantileTransformer(output_distribution='normal')
        self.scaler_s = StandardScaler()

    def fit(self, X, y=None):
        self.scaler_q.fit(X[:, self.quantile_idx])
        self.scaler_s.fit(X[:, self.standard_idx])
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        X_trans = np.zeros_like(X)
        X_trans[:, self.quantile_idx] = self.scaler_q.transform(X[:, self.quantile_idx])
        X_trans[:, self.standard_idx] = self.scaler_s.transform(X[:, self.standard_idx])
        return X_trans

    def inverse_transform(self, X):
        X_inv = np.zeros_like(X)
        X_inv[:, self.quantile_idx] = self.scaler_q.inverse_transform(X[:, self.quantile_idx])
        X_inv[:, self.standard_idx] = self.scaler_s.inverse_transform(X[:, self.standard_idx])
        return X_inv


#------------------------------------------------------------------------------
# Retreive the saved paramters of MLP

# Define paths (must match the saving cell)
model_save_path = '/content/drive/MyDrive/Data/mlp_model_weights.pth'
scaler_x_path = '/content/drive/MyDrive/Data/scaler_X.pkl'
scaler_y_path = '/content/drive/MyDrive/Data/scaler_y.pkl'

# Load the scalers
scaler_X = joblib.load(scaler_x_path)
scaler_y = joblib.load(scaler_y_path)
print(f"Scalers successfully loaded from '{scaler_x_path}' and '{scaler_y_path}'")

# Initialize the model structure
# We extract the input/output dimensions from the scalers
INPUT_DIM = scaler_X.n_features_in_
OUTPUT_DIM = scaler_y.n_features_in_
HIDDEN_LAYERS = 5
NEURONS_PER_LAYER = 512

model = TunableMLP(
    input_dim=INPUT_DIM,
    output_dim=OUTPUT_DIM,
    hidden_layers=HIDDEN_LAYERS,
    neurons_per_layer=NEURONS_PER_LAYER
)

# Load the model weights
model.load_state_dict(torch.load(model_save_path,  weights_only=True))
print(f"Model parameters successfully loaded ")

#-------------------------------------------------------------------------------
# Set random seeds for reproducibility
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

#-------------------------------------------------------------------------------
# Identify Input (Frequency Response) and Output (Component Values) columns
# Outputs: L, Cs, Cp columns
output_cols = ['L_0', 'L_1', 'L_2', 'Cs_0', 'Cs_1', 'Cs_2', 'Cp_0', 'Cp_1', 'Cp_2']

#------------------------------------------------------------------------------
# Load piecewise continuous filter responses for RL training
# Define the path to the CSV file in your Google Drive
file_path = '/content/drive/MyDrive/Data/piecewise_filter_responses_small.csv'

# Load the CSV file into a pandas DataFrame
df_piecewise = pd.read_csv(file_path)

# Display the first few rows of the DataFrame to verify it loaded correctly
display(df_piecewise.head())

<font color="blue"> **Step 5:** Define PPO environment

<font color="blue">1- PPO Environment includes methods for initialization, taking a step and reset

<font color="blue">2- A few methods are also defined for visualization of training.

<font color="blue">3- The Last method calcualtes the error of initial design from MLP and optimized design by PPO for samples of training and test sets.

In [ ]:
 #1. Define the Custom RL Environment
class FilterOptEnv(gym.Env):
    def __init__(self, initial_guesses, target_mags, freqs, acculamted_steps, max_steps):
        super(FilterOptEnv, self).__init__()
        self.target_mags = target_mags
        self.freqs = freqs
        self.initial_guesses = initial_guesses
        self.randomseed=42

        # Action space: additive change to log10 parameters (+/- 0.0414 decades per step -> ~10% change)
        self.action_space = spaces.Box(low=-0.0414, high=0.0414, shape=(9,), dtype=np.float32)

        # Observation space: 9 log params + (mag - target_mag)
        obs_dim = 9 + len(self.freqs)
        # Update observation space to reflect more realistic bounds based on clipped mag values
        self.observation_space = spaces.Box(low=-50.0, high=50.0, shape=(obs_dim,), dtype=np.float32)

        # Normalization factor for frequency response difference
        self.freq_resp_diff_norm_factor = 1.0
        # Weight for the normalized frequency response difference in the observation
        self.mag_diff_weight = 1.0 # Default weight, can be configured

        # search parameters
        self.acculamted_steps = acculamted_steps
        self.max_steps = max_steps
        self.current_step = 0
        self.best_reward = -np.inf
        self.best_params = self.initial_guesses[0].copy()

        # Sequential tracking
        self.episode_count = 0

        # Current episode state
        self.current_target_idx = 0
        self.current_log_params = np.log10(np.maximum(self.initial_guesses[0], 1e-15))

    def step(self, action):
        self.current_step += 1

        # Apply action: additive change in log space
        self.current_log_params = self.current_log_params + action
        # Prevent extreme negative values
        self.current_log_params = np.maximum(self.current_log_params, -15.0)

        # Convert back to linear space for simulation
        linear_params = 10 ** self.current_log_params

        # Unpack parameters
        L_opt = linear_params[0:3]
        Cs_opt = linear_params[3:6]
        Cp_opt = linear_params[6:9]

        # Simulate
        mag = ACsim(L_opt, Cs_opt, Cp_opt, self.freqs, render_mode=False)

        # Clamp mag to meaningful range for observation
        # Using val_FR_hi (5) as upper bound for safety, and -150 for lower bound.
        mag = np.clip(mag, -150.0, 5.0)

        # Calculate Reward ,which is negative of error
        current_target_mag = self.target_mags[self.current_target_idx]

        # Use MAE instead of MSE for shape error
        # +1e-8 to avoid division by zero
        shape_error = np.mean(np.abs((mag - current_target_mag) / (np.abs(current_target_mag) + 1e-8)))

        # Our goal is to minimize error/penalty, so reward is negative error
        reward = -shape_error

        # No reward clipping: relying on VecNormalize to handle scaling

        # Track best parameters (across all goals, for general tracking)
        # Note: We track the LINEAR params so downstream evaluation cells work correctly
        if reward > self.best_reward:
            self.best_reward = reward
            self.best_params = linear_params.copy()

        done = self.current_step >= self.max_steps
        truncated = False

        # Form  observation by concatenating log params and the difference between mag and target
        mag_diff = mag - current_target_mag
        # Normalize and weight mag_diff as requested
        normalized_mag_diff = mag_diff
        weighted_mag_diff = normalized_mag_diff

        obs = np.concatenate([self.current_log_params, weighted_mag_diff]).astype(np.float32)

        return obs, reward, done, truncated, {}

    def reset(self, seed=None, options=None):
        super().reset(seed=self.randomseed)
        self.current_step = 0

        # Sequentially select a target profile for this episode
        self.current_target_idx = self.episode_count % len(self.target_mags)
        self.episode_count += 1

        linear_params = self.initial_guesses[self.current_target_idx].copy()
        self.current_log_params = np.log10(np.maximum(linear_params, 1e-15))

        # Simulate initial state to get the initial magnitude response
        L_opt = linear_params[0:3]
        Cs_opt = linear_params[3:6]
        Cp_opt = linear_params[6:9]
        mag = ACsim(L_opt, Cs_opt, Cp_opt, self.freqs, render_mode=False)

        # Clamp mag to meaningful range for observation
        # Using val_FR_hi (5) as upper bound for safety, and -150 for lower bound.
        mag = np.clip(mag, -150.0, 5.0)

        # Form observation
        current_target_mag = self.target_mags[self.current_target_idx]
        mag_diff = mag - current_target_mag
        # Normalize and weight mag_diff as requested
        normalized_mag_diff = mag_diff
        weighted_mag_diff =  normalized_mag_diff

        obs = np.concatenate([self.current_log_params, weighted_mag_diff]).astype(np.float32)
        return obs, {}


#-------------------------------------------------------------------------------
# Visualization methods for RL training
import os
def plot_learning_curve(log_folder, title='PPO Learning Curve'):
    try:
        x, y = ts2xy(load_results(log_folder), 'timesteps')
        plt.figure(figsize=(10, 5))
        plt.plot(x, y, label='Episodic Reward', color='orange')
        plt.xlabel('Timesteps')
        plt.ylabel('Reward')
        plt.title(title)
        plt.grid(True)
        plt.legend()
        plt.show()
    except Exception as e:
        print(f"Could not plot learning curve: {e}")

log_dir = "/tmp/gym_normalized_fixed/"
tb_log_dir = "/tmp/tb_logs/" # Tensorboard directory
if os.path.exists(log_dir):
    shutil.rmtree(log_dir)
os.makedirs(log_dir, exist_ok=True)
os.makedirs(tb_log_dir, exist_ok=True)
#-------------------------------------------------------------------------------
# Custom Callback to log VecNormalize stats
class VecNormalizeCallback(BaseCallback):
    def __init__(self, verbose=0):
        super(VecNormalizeCallback, self).__init__(verbose)

    def _on_step(self) -> bool:
        if hasattr(self.training_env, 'obs_rms'):
            # Log the mean of the observation stats across all dimensions
            self.logger.record('vec_normalize/obs_rms_mean', np.mean(self.training_env.obs_rms.mean))
            self.logger.record('vec_normalize/obs_rms_var', np.mean(self.training_env.obs_rms.var))
        if hasattr(self.training_env, 'ret_rms'):
            self.logger.record('vec_normalize/ret_rms_mean', np.mean(self.training_env.ret_rms.mean))
        return True

#-------------------------------------------------------------------------------
# Plot RL paramters post training
class RLProgressPlotter:
    def __init__(self):
        pass

    def plot_all_metrics(self, sb3_log_dir_norm):
        try:
            progress_df = pd.read_csv(os.path.join(sb3_log_dir_norm, "progress.csv"))

            # --- 1. Plot Value and Policy Loss ---
            plt.figure(figsize=(14, 5))
            plt.subplot(1, 2, 1)
            if 'train/value_loss' in progress_df.columns:
                valid_data = progress_df.dropna(subset=['train/value_loss'])
                steps = valid_data['time/total_timesteps'] if 'time/total_timesteps' in valid_data.columns else range(len(valid_data))
                plt.plot(steps, valid_data['train/value_loss'], label='Value Loss', color='blue', marker='o', markersize=3)
            plt.xlabel('Timesteps')
            plt.ylabel('Value Loss')
            plt.title('Value Loss over Time')
            plt.grid(True)
            plt.legend()

            plt.subplot(1, 2, 2)
            if 'train/policy_gradient_loss' in progress_df.columns:
                valid_data = progress_df.dropna(subset=['train/policy_gradient_loss'])
                steps = valid_data['time/total_timesteps'] if 'time/total_timesteps' in valid_data.columns else range(len(valid_data))
                plt.plot(steps, valid_data['train/policy_gradient_loss'], label='Policy Loss', color='green', marker='o', markersize=3)
            plt.xlabel('Timesteps')
            plt.ylabel('Policy Loss')
            plt.title('Policy Gradient Loss over Time')
            plt.grid(True)
            plt.legend()
            plt.tight_layout()
            plt.show()

            # --- 2. Plot Entropy Loss, Explained Variance, and Approx KL ---
            plt.figure(figsize=(18, 5))
            plt.subplot(1, 3, 1)
            if 'train/entropy_loss' in progress_df.columns:
                valid_data = progress_df.dropna(subset=['train/entropy_loss'])
                steps = valid_data['time/total_timesteps'] if 'time/total_timesteps' in valid_data.columns else range(len(valid_data))
                plt.plot(steps, valid_data['train/entropy_loss'], label='Entropy Loss', color='purple', marker='o', markersize=3)
            plt.xlabel('Timesteps')
            plt.ylabel('Entropy Loss')
            plt.title('Entropy Loss over Time')
            plt.grid(True)
            plt.legend()

            plt.subplot(1, 3, 2)
            if 'train/explained_variance' in progress_df.columns:
                valid_data = progress_df.dropna(subset=['train/explained_variance'])
                steps = valid_data['time/total_timesteps'] if 'time/total_timesteps' in valid_data.columns else range(len(valid_data))
                plt.plot(steps, valid_data['train/explained_variance'], label='Explained Variance', color='red', marker='o', markersize=3)
            plt.xlabel('Timesteps')
            plt.ylabel('Explained Variance')
            plt.title('Explained Variance over Time')
            plt.grid(True)
            plt.legend()

            plt.subplot(1, 3, 3)
            if 'train/approx_kl' in progress_df.columns:
                valid_data = progress_df.dropna(subset=['train/approx_kl'])
                steps = valid_data['time/total_timesteps'] if 'time/total_timesteps' in valid_data.columns else range(len(valid_data))
                plt.plot(steps, valid_data['train/approx_kl'], label='Approx KL', color='brown', marker='o', markersize=3)
            plt.xlabel('Timesteps')
            plt.ylabel('Approx KL')
            plt.title('Approx KL over Time')
            plt.grid(True)
            plt.legend()
            plt.tight_layout()
            plt.show()

            # --- 3. Plot VecNormalize Stats & Clip Fraction ---
            plt.figure(figsize=(20, 5))

            plt.subplot(1, 4, 1)
            if 'vec_normalize/obs_rms_mean' in progress_df.columns:
                valid_data = progress_df.dropna(subset=['vec_normalize/obs_rms_mean'])
                steps = valid_data['time/total_timesteps'] if 'time/total_timesteps' in valid_data.columns else range(len(valid_data))
                plt.plot(steps, valid_data['vec_normalize/obs_rms_mean'], label='Obs RMS Mean', color='magenta', marker='o', markersize=3)
            plt.xlabel('Timesteps')
            plt.ylabel('Value')
            plt.title('Obs RMS Mean')
            plt.grid(True)

            plt.subplot(1, 4, 2)
            if 'vec_normalize/obs_rms_var' in progress_df.columns:
                valid_data = progress_df.dropna(subset=['vec_normalize/obs_rms_var'])
                steps = valid_data['time/total_timesteps'] if 'time/total_timesteps' in valid_data.columns else range(len(valid_data))
                plt.plot(steps, valid_data['vec_normalize/obs_rms_var'], label='Obs RMS Var', color='cyan', marker='o', markersize=3)
            plt.xlabel('Timesteps')
            plt.ylabel('Value')
            plt.title('Obs RMS Variance')
            plt.grid(True)

            plt.subplot(1, 4, 3)
            if 'vec_normalize/ret_rms_mean' in progress_df.columns:
                valid_data = progress_df.dropna(subset=['vec_normalize/ret_rms_mean'])
                steps = valid_data['time/total_timesteps'] if 'time/total_timesteps' in valid_data.columns else range(len(valid_data))
                plt.plot(steps, valid_data['vec_normalize/ret_rms_mean'], label='Ret RMS Mean', color='orange', marker='o', markersize=3)
            plt.xlabel('Timesteps')
            plt.ylabel('Value')
            plt.title('Return RMS Mean')
            plt.grid(True)

            plt.subplot(1, 4, 4)
            if 'train/clip_fraction' in progress_df.columns:
                valid_data = progress_df.dropna(subset=['train/clip_fraction'])
                steps = valid_data['time/total_timesteps'] if 'time/total_timesteps' in valid_data.columns else range(len(valid_data))
                plt.plot(steps, valid_data['train/clip_fraction'], label='Clip Fraction', color='black', marker='o', markersize=3)
            plt.xlabel('Timesteps')
            plt.ylabel('Fraction')
            plt.title('Clip Fraction')
            plt.grid(True)

            plt.tight_layout()
            plt.show()

        except Exception as e:
            print(f"Could not plot losses. Exception: {e}")

#-------------------------------------------------------------------------------
# Method to evaluate MAPE for MLP and PPO RL for test and training data
def evaluate_mape_subset(profiles, guesses, model, num_samples=200):
    eval_indices = np.random.choice(len(profiles), min(num_samples, len(profiles)), replace=False)
    rl_mapes = []
    mlp_mapes = []
    for idx in eval_indices:
        t_prof = profiles[idx]
        t_guess = guesses[idx]

        # MLP Baseline MAPE calculation
        mlp_mag_resp = ACsim(t_guess[0:3], t_guess[3:6], t_guess[6:9], FreqVec_MHz, render_mode=False)
        mlp_mape = np.mean(np.abs((t_prof - mlp_mag_resp) / (np.abs(t_prof) + 1e-8))) * 100
        mlp_mapes.append(mlp_mape)

        # RL Agent Optimization MAPE calculation
        eval_env = FilterOptEnv([t_guess], [t_prof], FreqVec_MHz, Acculamted_steps, Max_steps)
        v_env = DummyVecEnv([lambda: eval_env])
        n_env = VecNormalize(v_env, norm_obs=True, norm_reward=False, clip_obs=10., training=False)
        if using_vec_norm:
            n_env.obs_rms = norm_env.obs_rms

        obs = n_env.reset()
        for _ in range(Max_steps):
            action, _ = model.predict(obs, deterministic=True)
            obs, _, done, _ = n_env.step(action)
            if done.any(): break

        best_params = eval_env.best_params
        rl_mag_resp = ACsim(best_params[0:3], best_params[3:6], best_params[6:9], FreqVec_MHz, render_mode=False)
        rl_mape = np.mean(np.abs((t_prof - rl_mag_resp) / (np.abs(t_prof) + 1e-8))) * 100
        rl_mapes.append(rl_mape)

    return np.mean(rl_mapes), np.mean(mlp_mapes)

<font color="blue"> **Step 6:** Train PPO and save the results to a file

In [ ]:
#---------------------Initial setup ---------------------------------
import sys
import subprocess
import importlib

# Ensure tensorboard is explicitly installed in the exact active kernel executable
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorboard", "tensorboardX"])

# Reload the logger module so it stops using the cached 'SummaryWriter = None' state
import stable_baselines3.common.logger
importlib.reload(stable_baselines3.common.logger)

# Explicitly monkey-patch SummaryWriter into stable_baselines3.common.logger
from torch.utils.tensorboard import SummaryWriter
stable_baselines3.common.logger.SummaryWriter = SummaryWriter

# Re-import configure just to be safe
from stable_baselines3.common.logger import configure
print("stable_baselines3 logger successfully patched!")

# Create an instance of the RLProgressPlotter class
plotter = RLProgressPlotter()

import warnings

# Suppress DeprecationWarnings specifically from jupyter_client
warnings.filterwarnings("ignore", category=DeprecationWarning, module='jupyter_client.*')

# Optionally, suppress all DeprecationWarnings if the above isn't sufficient
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Suppress standard warnings
warnings.filterwarnings('ignore')

# Safely filter out PySpice spam without silencing the RL agent
class BlockPySpiceFilter(logging.Filter):
    def filter(self, record):
        return not record.name.startswith('PySpice')

# Apply the filter to all current logging handlers
for handler in logging.root.handlers:
    handler.addFilter(BlockPySpiceFilter())

active_profiles_data = []
# Extract 1000 piecewise profiles from the dataframe
print("Loading 1000 profiles from dataset...")
for i in range(1, 1001):
    prof_name = f"H(f) {i}"
    if prof_name in df_piecewise.columns:
        prof_data = df_piecewise[prof_name].values
        # Standardize Passband 2300-2400 for these generated profiles
        active_profiles_data.append((prof_data, 2300, 2400))


#------------ Initial estimates are from supervised learning -------------------
target_profiles= []
initial_guesses = []

# Run MLP to get the initial guess for each active profile
for prof, pb_start, pb_end in active_profiles_data:
    target_profiles.append(prof)

    # MLP Prediction
    # 1. Preprocess the input
    input_sample_scaled = scaler_X.transform(prof.reshape(1, -1))
    # 2. Run Inference
    model.eval()
    with torch.no_grad():
        input_tensor = torch.tensor(input_sample_scaled, dtype=torch.float32)
        prediction_scaled = model(input_tensor)
        prediction_scaled_np = prediction_scaled.cpu().numpy()
    # 3. Inverse Transform to get physical units
    prediction_log = scaler_y.inverse_transform(prediction_scaled_np).flatten()
    prediction_raw = 10 ** prediction_log

    L_pred_rl = np.zeros(3)
    Cs_pred_rl = np.zeros(3)
    Cp_pred_rl = np.zeros(3)

    for j, col_name in enumerate(output_cols):
        val_pred = prediction_raw[j]
        if col_name.startswith('L_'):
            L_pred_rl[int(col_name.split('_')[1])] = val_pred
        elif col_name.startswith('Cs_'):
            Cs_pred_rl[int(col_name.split('_')[1])] = val_pred
        elif col_name.startswith('Cp_'):
            Cp_pred_rl[int(col_name.split('_')[1])] = val_pred

    initial_guesses.append(np.concatenate((L_pred_rl, Cs_pred_rl, Cp_pred_rl)))

# ---------------------------- Train/Test Split --------------------------------
print("Splitting data into 90% Training and 10% Testing sets...")
train_targets, test_targets, train_guesses, test_guesses = train_test_split(
    target_profiles, initial_guesses, test_size=0.1, random_state=42
)


# ---------------------------- PPO Environment ---------------------------------
# 2. Setup the environment using ONLY the training profiles
base_env = FilterOptEnv(train_guesses, train_targets, FreqVec_MHz, Acculamted_steps, Max_steps)
# Add a wrapper to plot various metrics and monitor the progress of PPO training
monitor_env = Monitor(base_env, log_dir)
# Wrap with DummyVecEnv and VecNormalize
# PPO agent expects to receive an object that conforms to the VecEnv interface.
# In this case we have only one env thus the use of DummyVecEnv
vec_env = DummyVecEnv([lambda: monitor_env])
# This is another wrapper onthe top of vec_env. There are a total of four : base, monitor, vec, norm
norm_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.)

# Define architecture of two neural networks
net_depth = 4
net_width = 64
network_type = "Distinct"
policy_kwargs = dict(net_arch=dict(pi=[net_width] * net_depth, vf=[net_width] * net_depth))

# 3. Initialize a PPO Agent
model_rl = PPO(
    "MlpPolicy",
    norm_env,
    verbose=1,  # Set to 1 to allow standard output
    tensorboard_log=tb_log_dir,
    learning_rate=0.0005,
    batch_size=256,
    n_steps=512,
    gamma=0.83,
    clip_range=0.23,
    gae_lambda=0.83,
    max_grad_norm=0.64,
    ent_coef=0.0004,
    policy_kwargs=policy_kwargs,
    seed=42
)

# Print dynamic hyperparameters for consistency
print(f"\n{'='*60}")
using_vec_norm = 'norm_env' in locals() and isinstance(norm_env, VecNormalize)
is_normalized_str = " (with VecNormalize)" if using_vec_norm else "(without VecNormalize)"
print(f"  Training PPO{is_normalized_str}:\n    max_grad_norm={model_rl.max_grad_norm}\n    batch_size={model_rl.batch_size}\n    n_steps={model_rl.n_steps}\n    gamma={model_rl.gamma}\n    gae_lambda={model_rl.gae_lambda}\n    clip_range={model_rl.clip_range(0) if callable(model_rl.clip_range) else model_rl.clip_range}\n    ent_coef={model_rl.ent_coef}\n    learning_rate={model_rl.learning_rate}\n    seed={model_rl.seed}\n    vec_normalize={using_vec_norm}\n    network_type={network_type} (Actor & Critic have separate weights)\n    net_depth={net_depth}\n    net_width={net_width}")
print(f"{'='*60}\n")

# Set up the SB3 logger to capture internal metrics like Value/Policy loss
sb3_log_dir_norm = "/tmp/sb3_logs_norm/"
os.makedirs(sb3_log_dir_norm, exist_ok=True)
# Configured explicitly to remove 'stdout' while retaining 'csv' and 'tensorboard'
new_logger = configure(sb3_log_dir_norm, ["csv", "tensorboard"])
model_rl.set_logger(new_logger)

# ---  Custom Callback for Normalized Rewards ---
class NormalizedRewardCallback(BaseCallback):
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.norm_episode_rewards = []
        self.current_reward = 0

    def _on_step(self) -> bool:
        # self.locals['rewards'] contains the normalized rewards from VecNormalize
        norm_reward = self.locals['rewards'][0]
        self.current_reward += norm_reward

        # Check if the episode is done
        if self.locals['dones'][0]:
            self.norm_episode_rewards.append(self.current_reward)

            # Keep a moving average of the last 100 episodes to match ep_rew_mean
            if len(self.norm_episode_rewards) > 100:
                self.norm_episode_rewards.pop(0)

            # Log to TensorBoard
            self.logger.record('rollout/ep_rew_mean_normalized', np.mean(self.norm_episode_rewards))
            self.current_reward = 0

        # --- NEW: Log VecNormalize Statistics ---
        if hasattr(self.training_env, 'obs_rms') and self.training_env.obs_rms is not None:
            # Log the mean of the observation means and variances (since observation is an array)
            self.logger.record('vec_normalize/obs_rms_mean', np.mean(self.training_env.obs_rms.mean))
            self.logger.record('vec_normalize/obs_rms_variance', np.mean(self.training_env.obs_rms.var))

        if hasattr(self.training_env, 'ret_rms') and self.training_env.ret_rms is not None:
            self.logger.record('vec_normalize/ret_rms_mean', np.mean(self.training_env.ret_rms.mean))

        return True

# ------------------- START TENSORBOARD BEFORE TRAINING ------------------------
get_ipython().run_line_magic('load_ext', 'tensorboard')
# Point TensorBoard to the directory where the configured logger is actually writing
get_ipython().run_line_magic('tensorboard', '--logdir /tmp/sb3_logs_norm/')

#-------------------------- Train the RL model----------------------------------
print("Training Agent... (TensorBoard started! Check the interactive UI above)")
start_time = time.time()
# Pass the custom callback to learn() along with VecNormalizeCallback
model_rl.learn(total_timesteps=base_env.acculamted_steps, callback=[VecNormalizeCallback(), NormalizedRewardCallback()], tb_log_name="PPO_Run", progress_bar=True)
end_time = time.time()
training_duration = end_time - start_time
print(f"RL Training completed in: {training_duration:.2f} seconds")
# Save the trained model to Google Drive
model_rl.save('/content/drive/MyDrive/Data/ppo_filter_opt_model.zip')
print("Trained RL model saved to Google Drive.")
print("\nPlotting PPO Training Learning Curve...")
plot_learning_curve(log_dir, title='PPO Learning Curve (Normalized)')

# Call the plot_all_metrics method using the log directory defined in the training cell
print("Plotting additional RL metrics...")
plotter.plot_all_metrics(sb3_log_dir_norm)

# --- Evaluate MAPE on Train and Test Sets ---
train_rl_mape, train_mlp_mape = evaluate_mape_subset(train_targets, train_guesses, model_rl, num_samples=N_samples_eval)
test_rl_mape, test_mlp_mape = evaluate_mape_subset(test_targets, test_guesses, model_rl, num_samples=N_samples_eval)
print(f"\n=> Average Training MAPE - MLP Initial: {train_mlp_mape:.4f}%, RL Optimized: {train_rl_mape:.4f}%")
print(f"=> Average Test MAPE  - MLP Initial: {test_mlp_mape:.4f}%, RL Optimized: {test_rl_mape:.4f}%")
print("\n--- RL Optimization Finished ---")

<font color="blue"> **Step 7:** Compare the inital design by MLP with the optimized design by PPO for a sample of input frequency reponse

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Pick a random index from the test set
idx = np.random.randint(0, len(test_targets))
target_profile = test_targets[idx]
mlp_guess = test_guesses[idx]

print(f"Evaluating Test Sample Index: {idx}")

# 2. Get MLP Baseline Response & Calculate MAPE
mlp_resp = ACsim(mlp_guess[0:3], mlp_guess[3:6], mlp_guess[6:9], FreqVec_MHz, render_mode=False)
mlp_mape = np.mean(np.abs((target_profile - mlp_resp) / (np.abs(target_profile) + 1e-8))) * 100
print(f"MLP Baseline MAPE: {mlp_mape:.4f}%")

# 3. Get RL Optimized Response & Calculate MAPE
# Setup evaluation environment for this specific sample
eval_env_base = FilterOptEnv([mlp_guess], [target_profile], FreqVec_MHz, Acculamted_steps, Max_steps)
v_env = DummyVecEnv([lambda: eval_env_base])
eval_norm_env = VecNormalize(v_env, norm_obs=True, norm_reward=False, clip_obs=10., training=False)

# Copy normalization statistics from the training environment if available
if 'using_vec_norm' in locals() and using_vec_norm:
    eval_norm_env.obs_rms = norm_env.obs_rms

# Run the trained RL agent to refine the guess
obs = eval_norm_env.reset()
for _ in range(Max_steps):
    action, _ = model_rl.predict(obs, deterministic=True)
    obs, _, done, _ = eval_norm_env.step(action)
    if done.any():
        break

# Retrieve the best parameters found during the episode and simulate
rl_guess = eval_env_base.best_params
rl_resp = ACsim(rl_guess[0:3], rl_guess[3:6], rl_guess[6:9], FreqVec_MHz, render_mode=False)
rl_mape = np.mean(np.abs((target_profile - rl_resp) / (np.abs(target_profile) + 1e-8))) * 100
print(f"RL Optimized MAPE: {rl_mape:.4f}%")

# 4. Plot the results
fig, axes = plt.subplots(1, 2, figsize=(20, 7)) # Create a figure with two subplots

# --- Main Plot ----
# Plot Target
axes[0].plot(FreqVec_MHz, target_profile, label='Target Profile (Piecewise)', color='black', linestyle='--', linewidth=2)

# Plot MLP Baseline
axes[0].plot(FreqVec_MHz, mlp_resp, label=f'MLP Baseline Guess (MAPE: {mlp_mape:.2f}%)', color='red', alpha=0.7, linewidth=2)

# Plot RL Optimized
axes[0].plot(FreqVec_MHz, rl_resp, label=f'RL Optimized [PPO] (MAPE: {rl_mape:.2f}%)', color='blue', linewidth=2)

# Highlight the passband region
axes[0].axvspan(2300, 2400, color='green', alpha=0.1, label='Passband (2300-2400 MHz)')

axes[0].set_title(f"Filter Response Comparison - Test Sample {idx}", fontsize=14)
axes[0].set_xlabel("Frequency (MHz)", fontsize=12)
axes[0].set_ylabel("Magnitude Response (dB)", fontsize=12)
axes[0].set_ylim(-100, 5) # Setting reasonable limits based on typical filter responses
axes[0].grid(True, which="both", linestyle='--', alpha=0.6)
axes[0].legend(fontsize=12)

# --- Zoomed In-Band Plot ---
# Plot Target
axes[1].plot(FreqVec_MHz, target_profile, label='Target Profile (Piecewise)', color='black', linestyle='--', linewidth=2)

# Plot MLP Baseline
axes[1].plot(FreqVec_MHz, mlp_resp, label=f'MLP Baseline Guess (MAPE: {mlp_mape:.2f}%)', color='red', alpha=0.7, linewidth=2)

# Plot RL Optimized
axes[1].plot(FreqVec_MHz, rl_resp, label=f'RL Optimized [PPO] (MAPE: {rl_mape:.2f}%)', color='blue', linewidth=2)

# Highlight the passband region (for consistency, though it's the whole view now)
axes[1].axvspan(2300, 2400, color='green', alpha=0.1, label='Passband (2300-2400 MHz)')

axes[1].set_title(f"Zoomed In-Band Response - Test Sample {idx}", fontsize=14)
axes[1].set_xlabel("Frequency (MHz)", fontsize=12)
axes[1].set_ylabel("Magnitude Response (dB)", fontsize=12)
axes[1].set_xlim(2300, 2400) # Zoom to passband
axes[1].set_ylim(-5, 5) # Smaller vertical scale for in-band loss
axes[1].grid(True, which="both", linestyle='--', alpha=0.6)
axes[1].legend(fontsize=12)

plt.tight_layout()
plt.show()

<font color="blue"> **Step 8a:** Hyperparameter tuning (optional)

In [ ]:
# Install optuna if not already present
!pip install optuna

In [ ]:
import optuna

def optimize_ppo(trial):
    # Sample hyperparameters to tune
    learning_rate  = trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True)
    # batch_size     = trial.suggest_categorical('batch_size', [64, 128, 256])
    batch_size     = 256
    #n_steps        = trial.suggest_categorical('n_steps', [256, 512, 1024, 2048])
    n_steps = 1024
    gamma          = trial.suggest_float('gamma', 0.8, 0.99)
    gae_lambda     = trial.suggest_float('gae_lambda', 0.8, 0.99)
    max_grad_norm  = trial.suggest_float('max_grad_norm', 0.6, 0.95)
    clip_range     = trial.suggest_float('clip_range', 0.15, 0.35)
    #ent_coef       = trial.suggest_float('ent_coef', 0.0001, 0.01, log=True)
    ent_coef       = 0.001
    net_width      = trial.suggest_categorical('net_width', [64, 128, 256])
    net_depth      = trial.suggest_int('net_depth', 2, 4)
    # Define architecture of critic and actor NNs
    policy_kwargs = dict(net_arch=dict(pi=[net_width] * net_depth, vf=[net_width] * net_depth))

    # Setup Environment (Re-initialize for each trial)
    base_env = FilterOptEnv(train_guesses, train_targets, FreqVec_MHz, Acculamted_steps, Max_steps)
    vec_env = DummyVecEnv([lambda: base_env])
    norm_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.)

    # Initialize PPO Model
    model = PPO(
        "MlpPolicy",
        norm_env,
        learning_rate=learning_rate,
        batch_size=batch_size,
        n_steps=n_steps,
        gamma=gamma,
        gae_lambda=gae_lambda,
        clip_range=clip_range,
        max_grad_norm=max_grad_norm,
        ent_coef=ent_coef,
        policy_kwargs=policy_kwargs,
        verbose=0,
        seed=42
    )

    # Train Model for a larger number of timesteps to keep the search robust
    try:
        model.learn(total_timesteps=Acculamted_steps_HPR)
    except Exception as e:
        # Return infinity if the training crashes (e.g. gradient explosion)
        return float('inf')

    # Evaluate using MAPE on a larger validation subset (100 test samples)
    # evaluate_mape_subset is already defined in your common notebook
    val_rl_mape, _ = evaluate_mape_subset(test_targets, test_guesses, model, num_samples=N_samples_eval_HPR)

    return val_rl_mape

print("Starting Robust Optuna Hyperparameter Search for PPO...")
# Create a study object and specify the direction is 'minimize' (since we want lower MAPE)
study = optuna.create_study(direction="minimize")

# Run the optimization for 100 trials.
study.optimize(optimize_ppo, n_trials=100)

print("\n"+"="*40)
print("Optimization Finished!")
print("Best hyperparameters:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")
print(f"Best validation MAPE: {study.best_value:.4f}%")
print("="*40)


<font color="blue"> **Step 8b:** Visualization of hyperparameter tuning (optional)

In [ ]:
import optuna.visualization as vis

# Plot optimization history
fig_history = vis.plot_optimization_history(study)
fig_history.show()

# Plot parameter importances
fig_importances = vis.plot_param_importances(study)
fig_importances.show()

# Plot parallel coordinate
fig_parallel = vis.plot_parallel_coordinate(study)
fig_parallel.show()
